In [0]:
CREATE OR REPLACE TABLE workspace.default.service_gap_index_v2 AS

WITH base AS (
  SELECT
    z.zip_code,
    z.borough,

    z.total_311_rodent_record_count,
    z.resident_rat_report_count,
    z.direct_rat_sighting_count,
    z.attracting_condition_report_count,
    z.resident_mouse_report_count,
    z.inspector_signs_record_count,
    z.resident_rat_not_closed_count,
    z.resident_rat_closed_count,
    z.resident_rat_closed_within_60s_count,
    z.resident_rat_distinct_coordinate_count,
    z.violation_row_count,
    z.distinct_restaurant_count,
    z.distinct_inspection_count,
    z.restaurants_with_rat_violation,
    z.restaurants_with_mouse_violation,
    z.restaurants_with_rodent_violation,
    z.inspections_with_rodent_violation,

    CASE
      WHEN z.distinct_restaurant_count > 0
      THEN ROUND(
        100.0 * z.resident_rat_report_count
          / z.distinct_restaurant_count,
        2
      )
      ELSE NULL
    END AS resident_rat_reports_per_100_restaurants,

    CASE
      WHEN z.distinct_restaurant_count > 0
      THEN ROUND(
        100.0 * z.distinct_inspection_count
          / z.distinct_restaurant_count,
        2
      )
      ELSE NULL
    END AS inspections_per_100_restaurants,

    CASE
      WHEN z.distinct_restaurant_count < 20
        AND z.resident_rat_report_count < 200
      THEN 'Insufficient restaurant and resident-report samples'

      WHEN z.distinct_restaurant_count < 20
      THEN 'Insufficient restaurant sample'

      WHEN z.resident_rat_report_count < 200
      THEN 'Insufficient resident-report sample'

      ELSE 'Scoreable'
    END AS sample_status

  FROM workspace.default.zip_summary_v2 z
),

scoreable AS (
  SELECT
    *,

    PERCENT_RANK() OVER (
      ORDER BY resident_rat_reports_per_100_restaurants
    ) AS resident_report_percentile,

    PERCENT_RANK() OVER (
      ORDER BY inspections_per_100_restaurants
    ) AS inspection_activity_percentile

  FROM base
  WHERE sample_status = 'Scoreable'
),

scored AS (
  SELECT
    *,

    ROUND(
      50 + 50 * (
        resident_report_percentile
        - inspection_activity_percentile
      ),
      2
    ) AS service_gap_index

  FROM scoreable
)

SELECT
  b.zip_code,
  b.borough,

  b.total_311_rodent_record_count,
  b.resident_rat_report_count,
  b.direct_rat_sighting_count,
  b.attracting_condition_report_count,
  b.resident_mouse_report_count,
  b.inspector_signs_record_count,
  b.resident_rat_not_closed_count,
  b.resident_rat_closed_count,
  b.resident_rat_closed_within_60s_count,
  b.resident_rat_distinct_coordinate_count,

  b.violation_row_count,
  b.distinct_restaurant_count,
  b.distinct_inspection_count,
  b.restaurants_with_rat_violation,
  b.restaurants_with_mouse_violation,
  b.restaurants_with_rodent_violation,
  b.inspections_with_rodent_violation,

  b.resident_rat_reports_per_100_restaurants,
  b.inspections_per_100_restaurants,

  s.resident_report_percentile,
  s.inspection_activity_percentile,
  s.service_gap_index,

  b.sample_status,

  c.geocoded_record_count,
  c.distinct_coordinate_count,
  c.top_coordinate_share_percent,
  c.geocoded_resident_rat_report_count,
  c.resident_rat_distinct_coordinate_count
    AS geocoded_resident_rat_distinct_coordinate_count,
  c.top_resident_coordinate_share_percent,

  CASE
    WHEN c.geocoded_record_count >= 20
      AND c.top_coordinate_share_percent >= 50
    THEN 'High all-record coordinate concentration'
    ELSE 'No all-record concentration warning'
  END AS all_record_coordinate_warning,

  CASE
    WHEN c.geocoded_resident_rat_report_count >= 20
      AND c.top_resident_coordinate_share_percent >= 50
    THEN 'High resident-report coordinate concentration'
    ELSE 'No resident-report concentration warning'
  END AS resident_coordinate_warning

FROM base b

LEFT JOIN scored s
  ON b.zip_code = s.zip_code

LEFT JOIN workspace.default.zip_coordinate_concentration_v2 c
  ON b.zip_code = c.zip_code;

SELECT
  COUNT(*) AS total_zips,
  COUNT_IF(sample_status = 'Scoreable') AS scoreable_zips,
  COUNT_IF(service_gap_index IS NOT NULL) AS zips_with_scores,
  COUNT_IF(
    sample_status != 'Scoreable'
    AND service_gap_index IS NOT NULL
  ) AS invalid_scored_zips
FROM workspace.default.service_gap_index_v2;

SELECT
  zip_code,
  borough,
  resident_rat_report_count,
  inspector_signs_record_count,
  distinct_restaurant_count,
  distinct_inspection_count,
  resident_report_percentile,
  inspection_activity_percentile,
  service_gap_index,
  sample_status,
  all_record_coordinate_warning,
  resident_coordinate_warning
FROM workspace.default.service_gap_index_v2
WHERE zip_code IN (
  '10035',
  '11379',
  '11415',
  '11423',
  '11430'
)
ORDER BY zip_code;

SELECT
  zip_code,
  borough,
  resident_rat_report_count,
  distinct_restaurant_count,
  resident_report_percentile,
  inspection_activity_percentile,
  service_gap_index
FROM workspace.default.service_gap_index_v2
WHERE sample_status = 'Scoreable'
ORDER BY service_gap_index DESC
LIMIT 15;

SELECT
  COUNT_IF(closure_seconds < 0) AS negative_closure_records,
  MIN(closure_seconds) AS minimum_closure_seconds,
  MAX(closure_seconds) AS maximum_closure_seconds
FROM workspace.default.rat_clean_v2
WHERE resident_rat_report
  AND closure_seconds IS NOT NULL;